<a href="https://colab.research.google.com/github/gabrielajuncosa/banking-ml-ai-portfolio/blob/main/03_llm_and_genai/RAG_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical RAG & Information Extraction - How to use, and when (and why) it goes wrong
## Tutorial for Data Science & AI Educators
**by Gabriela Juncosa, PhD**

This notebook implements a simple Retrieval-Augmented Generation (RAG) system using cat facts as a knowledge base. Instead of relying on what a language model already knows, RAG first searches for relevant facts, then uses them to generate a grounded answer. The pipeline runs entirely in Google Colab with no API keys required, using BAAI/bge-base-en-v1.5 for semantic search and TinyLlama as the language model. Adapted from this [tutorial](https://huggingface.co/ngxson/demo_simple_rag_py/blob/main/demo.py) by ngxson.





## 0. Setup

1.  **Before we start:** enable the free GPU runtime so everything runs without API keys. Go to **Runtime** → **Change runtime type** → select **T4 GPU** → click **Save**.

2.   Run the cell below to install all required libraries. This may take a minute.

In [2]:
# LIBRARY IMPORTS
# !pip install transformers accelerate sentence-transformers requests # run once, then comment

import requests # fetch data from URLs — https://pypi.org/project/requests/
from sentence_transformers import SentenceTransformer # convert text to vectors — https://sbert.net/
from transformers import pipeline # run language models locally — https://huggingface.co/docs/transformers
import textwrap # wrap long text output for readability
import warnings, logging # suppress non-critical warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

## 1. What is RAG?

**Retrieval-Augmented Generation (RAG)** is a technique that improves LLM responses
by first *searching* a knowledge base for relevant information, then passing that
information to the model to generate a grounded answer.

Instead of relying solely on what the model learned during training, RAG feeds it
specific, retrievable facts at the moment of the query. This is especially useful
when working with domain-specific or up-to-date information.

**The flow of our system:**

<center>
Your question <br> ↓ <br>EMBEDDING_MODEL → finds relevant cat facts <br> ↓
<br>LANGUAGE_MODEL → reads those facts and writes an answer <br> ↓ <br>Response
</center>

> 💬 **Discussion:** LLMs are trained on general data and have a knowledge cutoff —
> they have no access to your organisation's internal documents, recent events, or
> domain-specific knowledge. **Have you encountered these limitations in practice?**
> Consider a recent use case where the model's response lacked the context your
> work requires — a policy, a dataset, a client, or an internal process it simply
> didn't know about. **What comes to mind?**

## 2. Load the Data

Our knowledge base is a list of facts about HR policies in Austria — one fact per line. Each line will become a *chunk*: a small piece of text the system can search through.

In a real-world RAG system, this could be a set of policy documents, research papers, or product manuals.

In [4]:
# Load dataset from Github
url = 'https://raw.githubusercontent.com/gabrielajuncosa/banking-ml-ai-portfolio/main/03_llm_and_genai/bank-hr-facts.txt'
response = requests.get(url)
data = response.text.splitlines()
print(f'Loaded {len(data)} entries')

Loaded 99 entries


In [5]:
# Let's look at the first few entries to understand the format
for i, fact in enumerate(data[:5]):
    print(textwrap.fill(f'{i+1}. {fact}', width=80, subsequent_indent='   '))
    # print(f'{i+1}. {fact}') # simple print without text wrapping

1. Employees in Austria are entitled to a minimum of 5 weeks (25 working days)
   of paid annual leave per year under the Urlaubsgesetz (Holiday Act).
2. After 25 years of service with the same employer, Austrian employees are
   entitled to 6 weeks (30 working days) of annual leave per year.
3. Unused vacation days in Austrian banks typically carry over to the following
   calendar year, but must be consumed within two years to avoid forfeiture.
4. In most Austrian banks, vacation requests must be submitted at least two
   weeks in advance and are subject to operational requirements.
5. Austrian labor law prohibits employers from denying vacation entirely; if a
   request is denied, an alternative date must be offered within the same leave
   year.


> 💬 Note that each fact is a single sentence. **Why might shorter, focused chunks work better than long paragraphs for retrieval?**
>
> Shorter chunks work better because cosine similarity compares the meaning of the entire chunk against the query. A long paragraph often contains multiple ideas — only one of which may be relevant. When that paragraph is retrieved, the irrelevant parts get passed to the language model as noise, which can dilute or confuse the answer. A focused, single-idea chunk gives the model exactly what it needs and nothing more.

## 3. Build the Vector Database

To search our knowledge base, we first need to convert each fact into a
**vector** — a list of numbers that captures its meaning. This is called
an *embedding*. Facts that are semantically similar will have similar
vectors, even if they use different words. This is what allows the system
to find relevant facts even when the query doesn't match word-for-word.

We use **BGE-base-en-v1.5** by BAAI — a lightweight, accurate embedding
model that runs entirely in Colab.

⚠️ BGE-base encodes surface vocabulary rather than deep query intent,
which can produce misleadingly high similarity scores for out-of-topic
queries that share common domain words with the corpus. This is a known
limitation of running a self-contained notebook on Colab's free tier;
production systems should consider `BAAI/bge-large-en-v1.5` or
`intfloat/e5-large-v2` for more robust retrieval.

In [6]:
# Load the embedding model
EMBEDDING_MODEL = SentenceTransformer('BAAI/bge-base-en-v1.5')
print('Embedding model loaded.')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


## 💡 Embeddings — what do they do?

When you pass text through this embedding model, it outputs a list of 768 numbers called a **vector**. These are not word counts or frequencies. A single number means nothing on its own.

One could think of the 768 numbers as **coordinates in a map of meaning**. Sentences with similar meaning end up close together in that space, even if they use different words. That is how the model can match "Are cats pink?" with a fact about colour perception in cats — no shared words, but similar meaning.

The 768 dimensions have no known meaning — they were never explicitly programmed and cannot be interpreted individually.They emerged automatically from training on large amounts of text. We trust the geometry that results.

*Let's run a quick test to see this in action.*

In [7]:
test = EMBEDDING_MODEL.encode("What is the current interest rate for a fixed mortgage product?")
print(test.shape)

(768,)


Now we encode every fact and store it in our vector database alongside the original text.

> 💬 Each embedding has hundreds of dimensions. **How do you think the model decides which numbers represent which aspects of meaning?**
>
> The model does not decide. There is no explicit mapping like "dimension 42 = colour" or "dimension 107 = animals." The dimensions emerge automatically through training — the model adjusts millions of numerical weights to minimise prediction errors across billions of text examples. What results is a high-dimensional space where meaning is encoded geometrically, but in a way that is not human-interpretable. No individual dimension has a fixed, readable meaning.

In [8]:
# Each element in the VECTOR_DB will be a tuple (chunk, embedding)
# The embedding is a list of floats, for example: [0.1, 0.04, -0.34, 0.21, ...]
VECTOR_DB = []

def add_chunk_to_database(chunk):
    embedding = EMBEDDING_MODEL.encode(chunk).tolist()
    VECTOR_DB.append((chunk, embedding))

for i, chunk in enumerate(data):
    add_chunk_to_database(chunk)
    if (i + 1) % 20 == 0:
      print(f'Added chunk {i+1}/{len(data)} to the database')

Added chunk 50/99 to the database


## 4. Search with Cosine Similarity

To find the most relevant facts for a question, we compare the question's vector
to every fact's vector using **cosine similarity**.

$$\text{cosine similarity} = \frac{A \cdot B}{\|A\| \|B\|}$$

Where **A** is the query vector, **B** is the fact vector, and the result is a
score between 0 and 1:
- A score of **1.0** means the vectors point in the same direction — very relevant
- A score of **0.0** means no similarity at all

This is how the system finds facts that are *semantically close* to the question,
even if they don't share the same words.

In [9]:
def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)

The `retrieve()` function below uses cosine similarity to rank all facts and return the top N most relevant ones.

In [10]:
def retrieve(query, top_n=3):
  query_embedding = EMBEDDING_MODEL.encode(query).tolist()
  # temporary list to store (chunk, similarity) pairs
  similarities = []
  for chunk, embedding in VECTOR_DB:
    similarity = cosine_similarity(query_embedding, embedding)
    similarities.append((chunk, similarity))
  # sort by similarity in descending order — higher score means more relevant
  similarities.sort(key=lambda x: x[1], reverse=True)
  # return the top N most relevant chunks
  return similarities[:top_n]

## 5. Ask a Question

Now we put it all together. The language model receives the retrieved facts
as context and is instructed to answer only based on that information —
not from its general training.

We use **TinyLlama** — a compact 1.1B parameter model that runs directly
on Colab's free T4 GPU.

⚠️ Due to its small size, TinyLlama may not reliably follow complex
instructions such as *"only answer from this context,"* and can default
to pre-trained general knowledge when retrieved context feels insufficient
— producing confident but hallucinated answers. This is a known limitation
of running a self-contained notebook on Colab's free tier; production RAG
systems should use a larger instruction-tuned model (7B+ parameters) for
reliable grounding.


In [11]:
# Load the language model
LANGUAGE_MODEL = pipeline('text-generation', model='TinyLlama/TinyLlama-1.1B-Chat-v1.0')
print('Language model loaded.')

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Language model loaded.


In [20]:
# Chatbot
input_query = input('Ask me a question: ')
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
  wrapped = textwrap.fill(f'(similarity: {similarity:.2f}) {chunk}', width=80, initial_indent=' - ', subsequent_indent='   ')
  print(wrapped)

instruction_prompt = f'''Answer ONLY based on the provided context below.
If the context does not contain enough information
to answer the question, respond exactly with:
"I don't have information on that topic in my knowledge base."
Do not use any outside knowledge.
{'\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}'''

# Generate response using TinyLlama
messages = f"<|system|>{instruction_prompt}</s><|user|>{input_query}</s><|assistant|>"
response = LANGUAGE_MODEL(messages, max_new_tokens=200, do_sample=True, temperature=0.7)

# print the response from the chatbot in real-time
print('Chatbot response:')
response_text = response[0]['generated_text'].split('<|assistant|>')[-1].strip()
print(textwrap.fill(response_text, width=80))

Ask me a question: How many vacation days do I get per year?
Retrieved knowledge:
 - (similarity: 0.67) Employees in Austria are entitled to a minimum of 5 weeks
   (25 working days) of paid annual leave per year under the Urlaubsgesetz
   (Holiday Act).
 - (similarity: 0.67) After 25 years of service with the same employer, Austrian
   employees are entitled to 6 weeks (30 working days) of annual leave per year.
 - (similarity: 0.65) Unused vacation days in Austrian banks typically carry
   over to the following calendar year, but must be consumed within two years to
   avoid forfeiture.
Chatbot response:
As per the information provided, Austrian employees are entitled to a minimum of
5 weeks (25 working days) of paid annual leave per year under the Urlaubsgesetz
(Holiday Act). However, the information provided did not specify the number of
vacation days employees in Austria are entitled to.


## 6. Try It Yourself

Now it's your turn to experiment. Before you run the cell, here is what the two
key parameters do:

**`top_n`** controls how many facts are retrieved from the database and passed to
the language model as context. With `top_n=3`, the three most similar facts are
retrieved. Increasing it gives the model more context to work with, but also
introduces more noise — facts that are less relevant. Decreasing it forces the
model to answer from fewer, more targeted facts.

**`temperature`** controls how predictable or creative the language model's
response is. Technically, it adjusts the probability distribution over the next
possible word at each step of generation. Intuitively:
- `0.1` — the model almost always picks the most likely next word. Responses are
  consistent and focused, but can feel repetitive or dry.
- `0.7` — a balance between coherence and variety. The default for most use cases.
- `1.0` — the model takes more risks with word choice. Responses feel more varied
  and creative, but can become less precise or go off-topic.

Try modifying the cell below:
- Change `top_n` to retrieve more or fewer facts — how does it affect the answer?
- Ask a question completely unrelated to cats — what happens?
- Change `temperature` to `0.1` or `1.0` — what do you notice about the response?

> 💬 Look at the similarity scores. **Do the retrieved facts seem relevant to your question? What happens if you ask something the documents don't cover?**

In [21]:
# Experiment here — change the question, top_n, or temperature
my_question = 'Can I work from home?'
my_top_n = 3
my_temperature = 0.7

retrieved = retrieve(my_question, top_n=my_top_n)

print('Retrieved knowledge:')
for chunk, similarity in retrieved:
    wrapped = textwrap.fill(
        f'(similarity: {similarity:.2f}) {chunk}',
        width=80, initial_indent=' - ', subsequent_indent='   '
    )
    print(wrapped)

instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
{'\n'.join([f' - {chunk}' for chunk, similarity in retrieved])}'''

messages = f'<|system|>{instruction_prompt}</s><|user|>{my_question}</s><|assistant|>'
response = LANGUAGE_MODEL(messages, max_new_tokens=200, do_sample=True, temperature=my_temperature)

print('\nChatbot response:')
response_text = response[0]['generated_text'].split('<|assistant|>')[-1].strip()
print(textwrap.fill(response_text, width=80))

Retrieved knowledge:
 - (similarity: 0.64) Austrian employees working from home are entitled to a
   cost contribution from their employer of up to €3 per day (maximum €300 per
   year) for home office expenses, which is tax-exempt.
 - (similarity: 0.60) Employees working from home in Austria are protected by
   the same accident insurance (AUVA) as in the office, covering accidents that
   occur during home office hours.
 - (similarity: 0.58) Remote work (Homeoffice) arrangements in Austria are
   governed by a specific legal framework introduced in 2021 under the
   Homeoffice-Paket.

Chatbot response:
Yes, you can work from home in Austria under certain circumstances and with
specific legal requirements.   Austrian employees working from home are entitled
to a cost contribution from their employer of up to €3 per day (maximum €300 per
year) for home office expenses, which is tax-exempt. This benefit is known as
the "Homeoffice-Paket" and was introduced in 2021 as part of a broader e